# Milestone 2: baseline generation, guidance and scheduler sweeps (Google Colab)

Runtime → Change runtime type → **T4 GPU** (or better), then *Runtime → Run all* and allow Google Drive access.

| Exp. | Script | What varies | Images |
|---|---|---|---|
| 1 | `01_baseline.py` | nothing: SD v1.5, DDIM, 30 steps, w = 7.5 on 200 COCO captions | 200 |
| 2 | `02_guidance_sweep.py` | guidance w ∈ {1, 3, 5, 7.5, 10, 15}; 32 prompts × 2 seeds | 384 |
| 3 | `03_scheduler_sweep.py` | DDIM, DDPM, PNDM, Euler, Euler-a, DPM++ × {10, 25, 50} steps; 24 prompts | 432 |

Prompts are COCO captions from Milestone 1's 5,000-pair manifest (`milestone1/results/pairs_manifest.csv`), in a fixed seeded order; prompt *i* uses seed *i* in every experiment, so only the varied setting changes. Every image is scored with **CLIP score** (ViT-L/14) against its caption.

**Run time:** about 1.5–2 hours on a T4. Generated images are written to `MyDrive/IE7615_Project2/milestone2_outputs` and existing images are skipped, so if Colab disconnects just *Run all* again and it continues where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IE7615_Project2'
except Exception as e:
    print('Google Drive not mounted, outputs stay in /content/outputs:', e)
    DRIVE = '/content/outputs'
REPO = 'https://github.com/anshuman-dataverse/text-to-image-generation.git'
os.makedirs(f'{DRIVE}/milestone2_outputs', exist_ok=True)
os.makedirs(f'{DRIVE}/data', exist_ok=True)

In [ ]:
%cd /content
!test -d project || git clone -q $REPO project
# fetch + hard reset: Colab-generated results/ must not block updating to the latest code
!cd project && git remote set-url origin $REPO && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
!pip -q install -r project/requirements.txt
# Colab's preinstalled torchao is too old for the latest diffusers (FqnToConfig import error); we don't use it
!pip -q uninstall -y torchao
# generated images on Drive (resumable); Milestone 1's processed COCO images, if present, for real-vs-generated figures
!rm -rf project/milestone2/outputs && ln -sfn "$DRIVE/milestone2_outputs" project/milestone2/outputs
!rm -rf project/milestone1/data && ln -sfn "$DRIVE/data" project/milestone1/data
!test -d project/milestone1/data/processed/images && echo "COCO images found: real-vs-generated figures and real-image CLIP scores on" || echo "No Milestone 1 COCO images on Drive: figures show generated images only"

In [ ]:
%cd /content/project/milestone2/scripts

import json, subprocess
from pathlib import Path
from IPython.display import Image, display

def run(script, *args):
    """Run a milestone script, stream its output here and to ../runs/<script>.log, stop on failure."""
    log = Path('../runs') / (Path(script).stem + '.log')
    log.parent.mkdir(exist_ok=True)
    commit = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
    print('code version:', commit)
    with open(log, 'w') as f:
        f.write(f'code version: {commit}\n')
        p = subprocess.Popen(['python', '-u', script, *map(str, args)], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            f.write(line)
    if p.wait():
        raise RuntimeError(f'{script} failed (exit code {p.returncode}). The error is printed above and saved in {log}')

def show(*names):
    for n in names:
        path = Path('../screenshots') / n
        display(Image(str(path))) if path.exists() else print(f'missing: {path} (the script before this cell did not finish)')

## 1. Baseline over COCO captions (DDIM, 30 steps, w = 7.5)

The Milestone 1 settings applied to 200 COCO captions instead of 5 hand-written prompts. CLIP score = 100 · max(cos(image, caption), 0) with CLIP ViT-L/14; when Milestone 1's processed COCO images are on Drive, the real photo is scored against the same caption as a reference.

In [ ]:
run('01_baseline.py')

In [ ]:
show('baseline_gallery.png', 'baseline_clip_hist.png', 'baseline_clip_extremes.png')

## 2. Classifier-free guidance scale

Same 32 prompts and seeds at w ∈ {1, 3, 5, 7.5, 10, 15}. Each prompt has 2 seeds, which gives a diversity measure: 1 − cosine similarity of the two images' CLIP embeddings. w = 1 turns guidance off (conditional model only). Expected: alignment rises with w then flattens, while diversity drops and colours saturate at high w.

In [ ]:
run('02_guidance_sweep.py')

In [ ]:
for s in json.load(open('../results/guidance_sweep_summary.json'))['per_scale']:
    print(f"w={s['guidance']:>4}  CLIP {s['clip_score']:.2f} ± {s['clip_score_ci95']:.2f}  "
          f"diversity {s.get('diversity', float('nan')):.3f}  saturation {s['saturation']:.3f}  {s['seconds_per_image']} s/img")
show('guidance_curves.png', 'guidance_grid.png')

## 3. Scheduler × sampling steps

Six schedulers (built from SD v1.5's trained noise schedule) at 10, 25 and 50 steps with w = 7.5. Besides CLIP score and time per image, we measure how far each image is from the same scheduler's 50-step image: deterministic samplers (DDIM, PNDM, Euler, DPM++) should converge, while the stochastic ones (DDPM, Euler-a) inject fresh noise each step and end on a different image.

In [ ]:
run('03_scheduler_sweep.py')

In [ ]:
for s in json.load(open('../results/scheduler_sweep_summary.json'))['per_config']:
    print(f"{s['scheduler']:>8} {s['steps']:>3} steps  CLIP {s['clip_score']:.2f} ± {s['clip_score_ci95']:.2f}  "
          f"Δpixel→50 {s['pixel_diff_to_50']}  {s['seconds_per_image']} s/img")
show('scheduler_heatmaps.png', 'scheduler_clip_vs_steps.png', 'scheduler_grid.png')

## 4. Save outputs to Drive
`milestone2_outputs.zip` holds results/, screenshots/ and runs/ (not the ~1,000 generated images, which stay in `MyDrive/IE7615_Project2/milestone2_outputs`). Download it, unzip into `milestone2/` of the local repo and commit.

In [ ]:
%cd /content/project/milestone2
!zip -qr "$DRIVE/milestone2_outputs.zip" results screenshots runs
!ls -la "$DRIVE"